In [2]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/laptop_pricing_dataset_mod1.csv"

In [3]:
import pandas as pd
import plotly as plt
import os
from IPython.display import display
import ipywidgets as widgets

In [4]:
import requests

def download(url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"Downloaded {filename} successfully.")
    else:
        print(f"Failed to fetch {url}. Status: {response.status_code}")


download(URL, "dataset.csv")


Downloaded dataset.csv successfully.


In [5]:
df = pd.read_csv(URL)
column_with_mission_values = df.columns[df.isnull().any()]
missing_rows = df[df.isnull().any(axis=1)]
display(column_with_mission_values)
display(missing_rows)

Index(['Screen_Size_cm', 'Weight_kg'], dtype='object')

,Unnamed: 0,Manufacturer,Category,Screen,GPU,OS,CPU_core,Screen_Size_cm,CPU_frequency,RAM_GB,Storage_GB_SSD,Weight_kg,Price
29,29,HP,3,IPS Panel,3,1,5,35.560,2.5,6,256,NaN,837
38,38,HP,3,IPS Panel,2,1,5,33.020,2.5,4,256,NaN,888
49,49,Dell,4,Full HD,2,1,5,33.782,1.6,8,256,NaN,1777
61,61,Dell,3,Full HD,1,2,7,39.624,2.7,16,256,NaN,1142
68,68,Dell,3,Full HD,1,2,7,39.624,1.8,8,256,NaN,934
120,120,Dell,4,Full HD,2,1,5,NaN,1.6,8,256,1.42,2340
151,151,Dell,5,Full HD,3,1,7,NaN,2.8,8,256,2.06,2240
187,187,Samsung,4,Full HD,2,1,7,NaN,2.7,8,256,1.31,2031
230,230,Dell,4,Full HD,2,1,5,NaN,2.5,8,256,1.36,1870


In [6]:
most_frequent_value = df['Screen_Size_cm'].mode()[0]
# df['Screen_Size_cm'].fillna(most_frequent_value, inplace=True) # Here inplace=True will be deprecated

mean_weight = df['Weight_kg'].mean()
# df['Weight_kg'].fillna(mean_weight, inplace=True) # Here inplace=True will be deprecated

df.fillna({
    'Screen_Size_cm' : most_frequent_value,
    'Weight_kg': mean_weight
})

display(df.isnull().sum())
display(df.dtypes)

Unnamed: 0        0
Manufacturer      0
Category          0
Screen            0
GPU               0
OS                0
CPU_core          0
Screen_Size_cm    4
CPU_frequency     0
RAM_GB            0
Storage_GB_SSD    0
Weight_kg         5
Price             0
dtype: int64

Unnamed: 0          int64
Manufacturer       object
Category            int64
Screen             object
GPU                 int64
OS                  int64
CPU_core            int64
Screen_Size_cm    float64
CPU_frequency     float64
RAM_GB              int64
Storage_GB_SSD      int64
Weight_kg         float64
Price               int64
dtype: object

In [7]:
# change data type
df['Screen_Size_cm'] = df['Screen_Size_cm'].astype(float)
df['Weight_kg'] = df['Weight_kg'].astype(float)
display(df.dtypes)


Unnamed: 0          int64
Manufacturer       object
Category            int64
Screen             object
GPU                 int64
OS                  int64
CPU_core            int64
Screen_Size_cm    float64
CPU_frequency     float64
RAM_GB              int64
Storage_GB_SSD      int64
Weight_kg         float64
Price               int64
dtype: object

In [8]:
# Convert Screen_Size_cm to inches and Weight_kg to pounds
df['Screen_Size_cm'] = df['Screen_Size_cm'] / 2.54
df['Weight_kg'] = df['Weight_kg'] * 2.20462

# Rename columns using dictionary-style with inplace=True
df.rename({
    'Screen_Size_cm': 'Screen_Size_inch',
    'Weight_kg': 'Weight_pounds'
}, axis=1, inplace=True)

# Optional: View updated DataFrame
display(df.head())


,Unnamed: 0,Manufacturer,Category,Screen,GPU,OS,CPU_core,Screen_Size_inch,CPU_frequency,RAM_GB,Storage_GB_SSD,Weight_pounds,Price
0,0,Acer,4,IPS Panel,2,1,5,14.0,1.6,8,256,3.527392,978
1,1,Dell,3,Full HD,1,1,3,15.6,2.0,4,256,4.850164,634
2,2,Dell,3,Full HD,1,1,7,15.6,2.7,8,256,4.850164,946
3,3,Dell,4,IPS Panel,2,1,5,13.3,1.6,8,128,2.689636,1244
4,4,HP,4,Full HD,2,1,7,15.6,1.8,8,256,4.210824,837


In [9]:
# Normalize the content under 'CPU_frequency' with respect to its maximum value
max_value = df['CPU_frequency'].max()
df['CPU_frequency'] = df['CPU_frequency'] / max_value
display(df.head())

,Unnamed: 0,Manufacturer,Category,Screen,GPU,OS,CPU_core,Screen_Size_inch,CPU_frequency,RAM_GB,Storage_GB_SSD,Weight_pounds,Price
0,0,Acer,4,IPS Panel,2,1,5,14.0,0.551724,8,256,3.527392,978
1,1,Dell,3,Full HD,1,1,3,15.6,0.689655,4,256,4.850164,634
2,2,Dell,3,Full HD,1,1,7,15.6,0.931034,8,256,4.850164,946
3,3,Dell,4,IPS Panel,2,1,5,13.3,0.551724,8,128,2.689636,1244
4,4,HP,4,Full HD,2,1,7,15.6,0.620690,8,256,4.210824,837


In [10]:
# 1. Create indicator variables using pd.get_dummies with desired prefix
df1 = pd.get_dummies(df['Screen'], prefix='Screen')

# 2. Append df1 to the original DataFrame
df = pd.concat([df, df1], axis=1)

# 3. Drop the original 'Screen' column
df.drop('Screen', axis=1, inplace=True)

# Optional: View updated DataFrame
display(df.head())


,Unnamed: 0,Manufacturer,Category,GPU,OS,CPU_core,Screen_Size_inch,CPU_frequency,RAM_GB,Storage_GB_SSD,Weight_pounds,Price,Screen_Full HD,Screen_IPS Panel
0,0,Acer,4,2,1,5,14.0,0.551724,8,256,3.527392,978,False,True
1,1,Dell,3,1,1,3,15.6,0.689655,4,256,4.850164,634,True,False
2,2,Dell,3,1,1,7,15.6,0.931034,8,256,4.850164,946,True,False
3,3,Dell,4,2,1,5,13.3,0.551724,8,128,2.689636,1244,False,True
4,4,HP,4,2,1,7,15.6,0.620690,8,256,4.210824,837,True,False


In [13]:
# Prompt the user for the exchange rate
exchange_rate = float(input("Enter the exchange rate from USD to Euro: "))

# Convert Price from USD to Euro
df['Price_Euro'] = df['Price'] * exchange_rate

# Display the updated DataFrame
display(df.head())


,Unnamed: 0,Manufacturer,Category,GPU,OS,CPU_core,Screen_Size_inch,CPU_frequency,RAM_GB,Storage_GB_SSD,Weight_pounds,Price,Screen_Full HD,Screen_IPS Panel,Price_Euro
0,0,Acer,4,2,1,5,14.0,0.551724,8,256,3.527392,978,False,True,831.30
1,1,Dell,3,1,1,3,15.6,0.689655,4,256,4.850164,634,True,False,538.90
2,2,Dell,3,1,1,7,15.6,0.931034,8,256,4.850164,946,True,False,804.10
3,3,Dell,4,2,1,5,13.3,0.551724,8,128,2.689636,1244,False,True,1057.40
4,4,HP,4,2,1,7,15.6,0.620690,8,256,4.210824,837,True,False,711.45


In [12]:

# Create a float input box
exchange_rate_widget = widgets.FloatText(
    value=0.9,  # default value
    description='Exchange Rate:',
    step=0.01
)

# Button to trigger conversion
button = widgets.Button(description="Convert Price")

# Function to execute on button click
def convert_price(b):
    df['Price_Euro'] = df['Price'] * exchange_rate_widget.value
    display(df.head())

# Connect the button to the function
button.on_click(convert_price)

# Display widgets
display(exchange_rate_widget, button)


FloatText(value=0.9, description='Exchange Rate:', step=0.01)

Button(description='Convert Price', style=ButtonStyle())

,Unnamed: 0,Manufacturer,Category,GPU,OS,CPU_core,Screen_Size_inch,CPU_frequency,RAM_GB,Storage_GB_SSD,Weight_pounds,Price,Screen_Full HD,Screen_IPS Panel,Price_Euro
0,0,Acer,4,2,1,5,14.0,0.551724,8,256,3.527392,978,False,True,880.2
1,1,Dell,3,1,1,3,15.6,0.689655,4,256,4.850164,634,True,False,570.6
2,2,Dell,3,1,1,7,15.6,0.931034,8,256,4.850164,946,True,False,851.4
3,3,Dell,4,2,1,5,13.3,0.551724,8,128,2.689636,1244,False,True,1119.6
4,4,HP,4,2,1,7,15.6,0.620690,8,256,4.210824,837,True,False,753.3
